In [2]:
import torch

torch.__version__

'2.11.0+cu130'

In [3]:
from typing import Any
from dataclasses import dataclass
from pathlib import Path
import os
import logging as log
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import ValenceType
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
import torch
from torch import nn


ATOM_SDF_FILENAME = "Conformer3D_COMPOUND_CID_4(1).sdf"
BIOACTIVITY_FILENAME = "pubchem_cid_4_bioactivity.csv"

BIOASSAY_NAME_KEYWORDS = {
    "BioAssay_Name_Has_qHTS": ("qhts",),
    "BioAssay_Name_Has_IC50": ("ic50",),
    "BioAssay_Name_Has_Counter_Screen": ("counter screen",),
    "BioAssay_Name_Has_Estrogen": ("estrogen",),
    "BioAssay_Name_Has_Androgen": ("androgen",),
    "BioAssay_Name_Has_Cytochrome": ("cytochrome",),
    "BioAssay_Name_Has_Plasmodium": ("plasmodium",),
    "BioAssay_Name_Has_Yeast": ("yeast",),
}

TARGET_NAME_KEYWORDS = {
    "Target_Name_Has_Estrogen": ("estrogen",),
    "Target_Name_Has_Androgen": ("androgen",),
    "Target_Name_Has_Cytochrome": ("cytochrome",),
    "Target_Name_Has_Plasmodium": ("plasmodium",),
    "Target_Name_Has_Yeast": ("yeast",),
}

SOURCE_KEYWORDS = {
    "Bioassay_Source_Has_Tox21": ("tox21",),
    "Bioassay_Source_Has_ChEMBL": ("chembl",),
    "Bioassay_Source_Has_DTP_NCI": ("dtp", "nci"),
}

RANDOM_SEED = 42


def get_data_dir():
    data_directory = os.environ["DATA_DIR"]
    log.info(f"Data directory: {data_directory}")
    return data_directory


def resolve_data_path(filename: str) -> Path:
    return Path(get_data_dir()) / filename


def load_bioactivity_dataframe(filename: str) -> pd.DataFrame:
    return pd.read_csv(resolve_data_path(filename))


def load_sdf_molecules(filename: str) -> list[Chem.Mol]:
    return get_valid_molecules(str(resolve_data_path(filename)))


def get_valid_molecules(sdf_file_path: str) -> list[Chem.Mol]:
    with Chem.SDMolSupplier(sdf_file_path, removeHs=False) as supplier:
        return [mol for mol in supplier if mol is not None]


def extract_atom_feature_matrix(molecules: list[Chem.Mol]) -> pd.DataFrame:
    atom_data: list[dict[str, object]] = []

    for molecule_index, mol in enumerate(molecules):
        for atom in mol.GetAtoms():
            atom_data.append(
                {
                    "moleculeIndex": molecule_index,
                    "index": atom.GetIdx(),
                    "bondCount": atom.GetDegree(),
                    "charge": atom.GetFormalCharge(),
                    "implicitHydrogenCount": atom.GetNumImplicitHs(),
                    "totalHydrogenCount": atom.GetTotalNumHs(),
                    "atomicNumber": atom.GetAtomicNum(),
                    "symbol": atom.GetSymbol(),
                    "valency": atom.GetValence(which=ValenceType.EXPLICIT),
                    "isAromatic": atom.GetIsAromatic(),
                    "mass": atom.GetMass(),
                    "hybridization": str(atom.GetHybridization()),
                    "properties": atom.GetPropsAsDict(),
                }
            )

    atom_feature_df = pd.DataFrame(atom_data)
    if atom_feature_df.empty:
        raise ValueError(
            "Expected at least one atom when building the atom feature matrix"
        )

    return atom_feature_df


def encode_categories(series: pd.Series) -> pd.Series:
    values = series.astype("string").str.strip().fillna("Unknown")
    values = values.mask(values.eq(""), "Unknown")
    category_order = sorted(values.unique().tolist())
    mapping = {value: index for index, value in enumerate(category_order)}
    return values.map(mapping).astype(int)


def build_atom_feature_frame(
    filename: str = ATOM_SDF_FILENAME,
) -> tuple[pd.DataFrame, Chem.Mol]:
    molecules = load_sdf_molecules(filename)

    if not molecules:
        raise ValueError(f"No valid molecules were loaded from {filename}")

    molecule = molecules[0]
    atom_feature_df = extract_atom_feature_matrix([molecule]).copy()
    atom_feature_df["atomId"] = atom_feature_df["index"].astype(int) + 1
    atom_feature_df["hybridizationEncoded"] = encode_categories(
        atom_feature_df["hybridization"]
    )
    atom_feature_df["isAromaticInt"] = atom_feature_df["isAromatic"].astype(int)
    atom_feature_df["isHeavyAtom"] = (
        atom_feature_df["atomicNumber"].astype(int).gt(1).astype(int)
    )
    atom_feature_df["elementLabel"] = atom_feature_df["symbol"].astype(str)
    atom_feature_df["atomLabel"] = atom_feature_df["symbol"].astype(
        str
    ) + atom_feature_df["atomId"].astype(str)
    return atom_feature_df, molecule


def build_molecular_descriptors(
    atom_feature_df: pd.DataFrame, molecule: Chem.Mol
) -> dict[str, Any]:
    element_counts = atom_feature_df["symbol"].astype(str).value_counts().to_dict()
    chiral_centers = Chem.FindMolChiralCenters(molecule, includeUnassigned=True)

    return {
        "C_count": int(element_counts.get("C", 0)),
        "H_count": int(element_counts.get("H", 0)),
        "N_count": int(element_counts.get("N", 0)),
        "O_count": int(element_counts.get("O", 0)),
        "bond_count_total": int(molecule.GetNumBonds()),
        "mol_weight": float(atom_feature_df["mass"].astype(float).sum()),
        "has_chiral_center": int(len(chiral_centers) > 0),
    }


@dataclass(frozen=True)
class PreparedDataset:
    name: str
    task_type: str
    frame: pd.DataFrame
    feature_columns: list[str]
    target_column: str | None
    description: str
    class_names: list[str] | None = None

    def feature_matrix(self) -> np.ndarray:
        return self.frame.loc[:, self.feature_columns].to_numpy(dtype=np.float64)

    def target_vector(self) -> np.ndarray:
        if self.target_column is None:
            raise ValueError(f"Dataset {self.name} does not define a target column")

        return self.frame[self.target_column].to_numpy()

    def summary(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "task_type": self.task_type,
            "row_count": int(len(self.frame)),
            "feature_count": int(len(self.feature_columns)),
            "feature_columns": list(self.feature_columns),
            "target_column": self.target_column,
            "class_names": None if self.class_names is None else list(self.class_names),
            "description": self.description,
        }


@dataclass(frozen=True)
class SupervisedSplit:
    x_train: np.ndarray
    x_test: np.ndarray
    y_train: np.ndarray
    y_test: np.ndarray
    scaler: StandardScaler | None
    used_holdout_split: bool
    evaluation_note: str


def build_activity_value_regression_dataset(
    bioactivity_filename: str = BIOACTIVITY_FILENAME,
    atom_filename: str = ATOM_SDF_FILENAME,
) -> PreparedDataset:
    bioactivity_df = load_bioactivity_dataframe(bioactivity_filename)
    activity_value = pd.to_numeric(bioactivity_df["Activity_Value"], errors="coerce")
    retained_mask = activity_value.notna() & activity_value.gt(0)
    filtered_df = bioactivity_df.loc[retained_mask].copy()
    filtered_df["Activity_Value"] = activity_value.loc[retained_mask]
    atom_feature_df, molecule = build_atom_feature_frame(atom_filename)
    descriptor_map = build_molecular_descriptors(atom_feature_df, molecule)
    frame, feature_columns = build_bioactivity_model_frame(filtered_df, descriptor_map)

    return PreparedDataset(
        name="bioactivity-activity-value-regression",
        task_type="regression",
        frame=frame,
        feature_columns=feature_columns,
        target_column="Activity_Value",
        class_names=None,
        description="Positive numeric Activity_Value regression dataset using "
        "molecular descriptors and assay metadata.",
    )


def add_keyword_flags(
    frame: pd.DataFrame, column_name: str, definitions: dict[str, tuple[str, ...]]
) -> None:
    normalized = frame[column_name].astype("string").fillna("").str.lower()
    for feature_name, keywords in definitions.items():
        pattern = "|".join(keywords)
        frame[feature_name] = normalized.str.contains(pattern, regex=True).astype(int)


def has_non_empty_text(series: pd.Series) -> pd.Series:
    normalized = series.astype("string").fillna("").str.strip()
    return normalized.ne("").astype(int)


def build_bioactivity_model_frame(
    source_frame: pd.DataFrame, descriptor_map: dict[str, Any]
) -> tuple[pd.DataFrame, list[str]]:
    frame = source_frame.copy()
    frame["Aid_Type_Encoded"] = encode_categories(frame["Aid_Type"])
    frame["Activity_Type_Encoded"] = encode_categories(frame["Activity_Type"])
    frame["Bioassay_Data_Source_Encoded"] = encode_categories(
        frame["Bioassay_Data_Source"]
    )
    frame["Has_Dose_Response_Curve"] = pd.to_numeric(
        frame["Has_Dose_Response_Curve"], errors="coerce"
    ).fillna(0)
    frame["RNAi_BioAssay"] = pd.to_numeric(
        frame["RNAi_BioAssay"], errors="coerce"
    ).fillna(0)
    frame["Taxonomy_ID_Numeric"] = pd.to_numeric(
        frame["Taxonomy_ID"], errors="coerce"
    ).fillna(-1)
    frame["Target_Taxonomy_ID_Numeric"] = pd.to_numeric(
        frame["Target_Taxonomy_ID"], errors="coerce"
    ).fillna(-1)
    frame["Has_Protein_Accession"] = has_non_empty_text(frame["Protein_Accession"])
    frame["Has_Gene_ID"] = has_non_empty_text(frame["Gene_ID"])
    frame["Has_PMID"] = has_non_empty_text(frame["PMID"])
    frame["Has_Activity_Value"] = has_non_empty_text(frame["Activity_Value"])

    for key, value in descriptor_map.items():
        frame[key] = value

    add_keyword_flags(frame, "BioAssay_Name", BIOASSAY_NAME_KEYWORDS)
    add_keyword_flags(frame, "Target_Name", TARGET_NAME_KEYWORDS)
    add_keyword_flags(frame, "Bioassay_Data_Source", SOURCE_KEYWORDS)

    feature_columns = [
        "C_count",
        "H_count",
        "N_count",
        "O_count",
        "bond_count_total",
        "mol_weight",
        "has_chiral_center",
        "Aid_Type_Encoded",
        "Activity_Type_Encoded",
        "Bioassay_Data_Source_Encoded",
        "Has_Dose_Response_Curve",
        "RNAi_BioAssay",
        "Taxonomy_ID_Numeric",
        "Target_Taxonomy_ID_Numeric",
        "Has_Protein_Accession",
        "Has_Gene_ID",
        "Has_PMID",
        "Has_Activity_Value",
        *BIOASSAY_NAME_KEYWORDS,
        *TARGET_NAME_KEYWORDS,
        *SOURCE_KEYWORDS,
    ]
    return frame, feature_columns


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else None,
    }


def can_build_holdout_split(task_type: str, y_values: np.ndarray) -> bool:
    if len(y_values) < 8:
        return False

    if task_type != "classification":
        return True

    _, counts = np.unique(y_values, return_counts=True)
    return len(counts) > 1 and int(counts.min()) >= 2


def build_supervised_split(
    dataset: PreparedDataset,
    *,
    test_size: float = 0.3,
    random_state: int = RANDOM_SEED,
    scale_features: bool = True,
) -> SupervisedSplit:
    x_values = dataset.feature_matrix()
    y_values = dataset.target_vector()
    used_holdout_split = can_build_holdout_split(dataset.task_type, y_values)

    if used_holdout_split:
        stratify = y_values if dataset.task_type == "classification" else None
        x_train, x_test, y_train, y_test = train_test_split(
            x_values,
            y_values,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )
        evaluation_note = "Evaluation uses a deterministic holdout split."
    else:
        x_train = x_values
        x_test = x_values
        y_train = y_values
        y_test = y_values
        evaluation_note = "Dataset is too small for a stable holdout split, so evaluation is reported on the training rows."

    scaler = None
    if scale_features:
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_test = scaler.transform(x_test)

    return SupervisedSplit(
        x_train=x_train,
        x_test=x_test,
        y_train=y_train,
        y_test=y_test,
        scaler=scaler,
        used_holdout_split=used_holdout_split,
        evaluation_note=evaluation_note,
    )

In [4]:
def run_torch_regression(dataset: PreparedDataset, epochs: int = 300) -> dict[str, Any]:
    split = build_supervised_split(dataset)
    x_train = torch.tensor(split.x_train, dtype=torch.float32)
    y_train = torch.tensor(
        split.y_train.astype(np.float32).reshape(-1, 1), dtype=torch.float32
    )
    x_eval = torch.tensor(split.x_test, dtype=torch.float32)
    model = nn.Sequential(
        nn.Linear(x_train.shape[1], 16),
        nn.ReLU(),
        nn.Linear(16, 1),
    )
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=0.0)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        predictions = model(x_train)
        loss = loss_fn(predictions, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        predictions = model(x_eval).cpu().numpy().reshape(-1)

    return {
        "dataset": dataset.summary(),
        "evaluation_note": split.evaluation_note,
        "epochs": int(epochs),
        "metrics": regression_metrics(
            split.y_test.astype(np.float64), predictions.astype(np.float64)
        ),
    }

In [ ]:
regression_dataset = build_activity_value_regression_dataset()
run_torch_regression(regression_dataset)